# GST Series and Pixel

## Initializing

In [2]:
import ee
import plotting as pl
import gst_series as gs
import gst_pixel as gp
import io_param_functions as ipf

In [ ]:
ee.Authenticate()
GEE_PROJECT = "my-hello-world-508910"  # replace with your own project id
ee.Initialize(project=GEE_PROJECT)

In [ ]:
ROI = [[81.768322, 30.198924], [81.785316, 30.200185], [81.782656, 30.20241], [81.780596, 30.2079], [81.780252, 30.215613], [81.780424, 30.223327], [81.778965, 30.228741], [81.776733, 30.231707], [81.774759, 30.231781], [81.771927, 30.225403], [81.77021, 30.216503], [81.770382, 30.211237], [81.770296, 30.206416], [81.766691, 30.202707], [81.765575, 30.199962], [81.768322, 30.198924]]
AOI = ee.Geometry.Polygon(ROI)

In [ ]:
START = "2014-09-01"
END = "2026-12-31"
YEARS = list(range(2015, 2027))

DEBUG = True
SAVE_PATH = "results"

## GST Series

In [ ]:
glacier_mask, mask_status = gs.get_glacier_mask(roi=AOI)
print("Glacier mask source:", mask_status)

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
bin_img = gs.get_dem_bins(dem)  # 50 bzw. BIN_M grosse Höhenstufen

In [ ]:
# Map = pl.show_dem_map(dem=dem, roi=AOI, glacier_mask=glacier_mask) # needs debugging / checking!
raw_gst = gs.download(bin_img, glacier_mask, AOI, START, END, debug=False)
gst_half_month = gs.process(raw_gst, debug=True)
ablation_trend = gs.ablation_trend(gst_half_month, debug=True)
gst_half_month.head()

In [ ]:
fig_gst_observations_1 = pl.plot_half_months(gst_half_month)
fig_gst_observations_2 = pl.plot_half_months_2(gst_half_month)

In [ ]:
fig_gst_err = pl.plot_gst_series(gst_half_month, errorbars=True)
fig_gst_noerr = pl.plot_gst_series(gst_half_month, errorbars=False)

In [ ]:
ipf.save_any_tables_and_figures(
    {"gst_raw_scene_bins": raw_gst,
     "gst_halfmonthly": gst_half_month,
     "gst_ablation_season": ablation_trend,
     "gst_obervations_1": fig_gst_observations_1,
     "gst_obervations_2": fig_gst_observations_2,
     "gst_err": fig_gst_err,
     "gst_noerr": fig_gst_noerr},
    save_path="results/gst_series/"
)

## GST Pixel

In [ ]:
scenes = gp.download_zone_scenes(AOI, glacier_mask, START, END, debug=True)
half_monthly = gp.process_zones(scenes, debug=True)

In [ ]:
winters = gp.winter_means(half_monthly)
print(gp.winter_trends(winters).round(3))

In [ ]:
pixel_arrays = gp.image_to_arrays(gp.pixel_statistics(AOI, glacier_mask, START, END), AOI)
fig = pl.plot_pixel_maps(pixel_arrays)

In [ ]:
point_series_df = gp.download_point_series({"centre": (81.774, 30.220)}, START, END, debug=True)

ipf.save_any_tables_and_figures(
    {"gst_scene_zone_sums": scenes,
     "gst_halfmonthly_winter": half_monthly,
     "gst_winter_means": winters,
     "gst_points": point_series_df,
     "gst_pixel_maps": fig},
    save_path="results/gst_pixel/"
)